# 02. 안정적인 prefix와 cache key

목표: timestamp, JSON key 순서, tool 순서가 의미상 같은 prompt의 hash를 어떻게 바꾸는지 확인합니다.

In [ ]:
import hashlib
import json

def cache_key(tools, system):
    prefix = json.dumps({"tools": tools, "system": system}, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(prefix.encode("utf-8")).hexdigest()[:16]

base_system = {"rules": ["JSON으로 답하기", "근거 없는 값 금지"]}
print("stable key:", cache_key([], base_system))

## Dynamic value를 prefix에 넣은 경우

Timestamp가 바뀌면 cache key도 바뀝니다. 변동값은 user message나 cache point 뒤로 옮겨야 합니다.

In [ ]:
system_a = {**base_system, "timestamp": "2026-07-22T09:00:00"}
system_b = {**base_system, "timestamp": "2026-07-22T09:00:01"}
print(cache_key([], system_a), cache_key([], system_b))
print("same prefix?", cache_key([], system_a) == cache_key([], system_b))

## JSON과 tool 순서를 정규화하기

최종 직렬화 순서가 같도록 key와 list를 정렬합니다.

In [ ]:
def canonicalize(value):
    if isinstance(value, dict):
        return {key: canonicalize(value[key]) for key in sorted(value)}
    if isinstance(value, list):
        normalized = [canonicalize(item) for item in value]
        if all(isinstance(item, dict) and "name" in item for item in normalized):
            return sorted(normalized, key=lambda item: item["name"])
        return normalized
    return value

def stable_cache_key(tools, system):
    payload = canonicalize({"tools": tools, "system": system})
    prefix = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(prefix.encode("utf-8")).hexdigest()[:16]

tools_a = [{"name": "search", "schema": {"q": "str"}}, {"name": "fetch", "schema": {"url": "str"}}]
tools_b = list(reversed(tools_a))
print(stable_cache_key(tools_a, base_system))
print(stable_cache_key(tools_b, base_system))
print("same after canonicalization?", stable_cache_key(tools_a, base_system) == stable_cache_key(tools_b, base_system))

## 회귀 검사

배포 전 대표 요청의 prefix hash를 snapshot으로 저장하면 의도치 않은 cache invalidation을 감지할 수 있습니다.

In [ ]:
expected = stable_cache_key(tools_a, base_system)
actual = stable_cache_key(tools_b, {"rules": ["JSON으로 답하기", "근거 없는 값 금지"]})
assert actual == expected
print("prefix regression check passed:", actual)